# Sprint 53 · Task B · Steps B3–B4 — GPU runs on Colab

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run cells 1 → 5.

| Cell | What it does | When |
|---|---|---|
| 1 | GPU check | every session |
| 2 | Upload zip(s) and unpack | every session |
| 3 | Install + start Ollama, download glm4 and qwen2.5:7b, record versions | every session (~3–6 min) |
| 4 | Stage 1 progress | every session |
| 5 | Define `save_checkpoint()` | every session |
| 6 | **Step B3** — smoke test (16 calls) + Stage 1 time estimate | Step B3 only |
| 7 | **Step B4** — Stage 1 screening run, 30-min chunks with checkpoint downloads | ⚠️ only after Step B3 is confirmed |
| 8 | Manual checkpoint | before closing the tab |

In [15]:
# 1. GPU check - should show a Tesla T4 with ~15 GB
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


**Cell 2 — what to upload** (one file at a time; the cell keeps asking until it has everything)
* **First time (Step B3):** your Step B2 checkpoint `sprint53_taskB_B2_<time>.zip`, then `sprint53_taskB_step3_code.zip`.
* **Later sessions:** only the newest `sprint53_taskB_run_<time>.zip` checkpoint.

In [16]:
# 2. Upload and unpack
import os, zipfile, glob
from google.colab import files
os.chdir('/content')

def unpack(path):
    with zipfile.ZipFile(path) as z:
        tops = {p.split('/')[0] for p in z.namelist() if p.strip('/')}
        if tops != {'sprint53'}:
            raise SystemExit(f'!! {path} does not contain a single sprint53/ folder (found {tops}).')
        z.extractall('/content')
    print('   unpacked', path)

have_data = lambda: os.path.exists('/content/sprint53/data/taskB/fewshot_exclude.json')
have_code = lambda: os.path.exists('/content/sprint53/taskB/run_taskb.py')
code_zips = []
while not (have_data() and have_code()):
    need = []
    if not have_data(): need.append('Step B2 checkpoint (sprint53_taskB_B2_<time>.zip) or newest run checkpoint')
    if not have_code(): need.append('Step B3 code (sprint53_taskB_step3_code.zip)')
    print('Please upload ONE file now:', ' / '.join(need))
    for name in files.upload():
        unpack(name)
        if 'step3_code' in name.lower(): code_zips.append(name)
        else:
            for c in code_zips: unpack(c)
os.chdir('/content/sprint53')
print('\nReady. Task B results so far:', sorted(os.path.basename(p) for p in glob.glob('results_taskB/*')) or 'none yet')


Ready. Task B results so far: ['B1_S1_rubric.jsonl', 'B1_S2_fewshot.jsonl', 'B1_S3_steps.jsonl', 'B1_S4_accflu.jsonl', 'B1_baseline.jsonl', 'B1_pairwise.jsonl', 'B1_repeat_T07.jsonl', 'environment.jsonl', 'progress_stage1.json', 'prompts_snapshot.json', 'smoke.jsonl', 'smoke_summary.json']


In [17]:
# 3. Install + start Ollama, download both judges, record GPU / Ollama / model versions
%cd /content/sprint53
!bash colab/setup_ollama.sh
!python taskB/run_taskb.py --check

/content
>> 1/4 System packages (zstd is needed by the Ollama installer, pciutils/lshw for GPU detection)
>> 2/4 Ollama already installed
>> 3/4 Starting the Ollama server in the background (log: /content/ollama.log)
{"version":"0.35.1"}
>> 4/4 Downloading judge models (same tags as Sprint 50)
success 
success 
NAME           ID              SIZE      MODIFIED               
glm4:latest    5b699761eca5    5.5 GB    Less than a second ago    
qwen2.5:7b     845dbda0ea48    4.7 GB    Less than a second ago    
>> Setup finished.
GPU            : Tesla T4, 15360 MiB
Ollama version : 0.35.1
Model glm4     : glm4:latest  (digest 5b699761eca5)
Model qwen     : qwen2.5:7b  (digest 845dbda0ea48)
OK - environment recorded in results/environment.jsonl
   (Task B copy: results_taskB/environment.jsonl - the Task A results/ folder is not touched)


In [18]:
# 4. Stage 1 progress  (before Step B4: 7 phases, TOTAL 2840, 0 done)
!python taskB/run_taskb.py --status --stage 1

phase             calls   done     % malformed
B1_baseline         632    632  100%         0  ✔
B1_S1_rubric        316    316  100%         0  ✔
B1_S2_fewshot       316    316  100%         0  ✔
B1_S3_steps         316    316  100%         1  ✔
B1_S4_accflu        316    316  100%         0  ✔
B1_pairwise         224    224  100%         0  ✔
B1_repeat_T07       720    108   15%         0  …
TOTAL              2840   2228   78%
Next phase to run: B1_repeat_T07


In [19]:
# 5. Checkpoint helper: zips /content/sprint53 (data, code, Task A + Task B results - not the models) and downloads it
import datetime, zipfile, os
from google.colab import files
def save_checkpoint():
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
    out = f'/content/sprint53_taskB_run_{stamp}.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, dirs, fnames in os.walk('/content/sprint53'):
            dirs[:] = [d for d in dirs if d not in ('node_modules', '.venv', '__pycache__')]
            for f in fnames:
                if f.endswith('.zip'): continue
                full = os.path.join(root, f); z.write(full, os.path.relpath(full, '/content'))
    print(f'Checkpoint: {out} ({os.path.getsize(out)/1e6:.1f} MB) - downloading...')
    files.download(out)
print('save_checkpoint() is ready')

save_checkpoint() is ready


## STEP B3 — smoke test
16 calls: the 2 baseline prompts, the 4 absolute strategy prompts and the 2 pairwise prompts, once per judge, on one
development item. Checks every answer can be read and measures speed per prompt; prints the Stage 1 time estimate.
Takes ~2–4 minutes, then downloads a checkpoint.

In [6]:
# 6. Smoke test (Step B3)
!python taskB/run_taskb.py --smoke
save_checkpoint()

>> Smoke test: 16 calls (8 prompts x 2 judges), models already loaded.
   [smoke] 16/16 calls  |  2.1 s/call  |  phase ETA 0m00s  |  session time left 59m27s

judge  prompt         parse         value tries   secs out tok    end
glm4   flat           ok               75     1   10.9      12   stop
glm4   mqm            ok               90     1    2.3      72   stop
glm4   flat_rubric    ok               85     1    0.5      12   stop
glm4   mqm_fewshot    ok               94     1    2.5      72   stop
glm4   mqm_steps      ok              100     1    3.1      91   stop
glm4   flat_accflu    ok             82.0     1    0.7      19   stop
glm4   pairwise       ok              tie     1    0.4      12   stop
glm4   pairwise_tie   ok              tie     1    4.1     133   stop
qwen   flat           ok               65     1    0.3       8   stop
qwen   mqm            ok               95     1    1.2      37   stop
qwen   flat_rubric    ok               65     1    0.4       8   stop
q

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## STEP B4 — Stage 1 screening run  ⚠️ do NOT run until Step B3 is confirmed
Runs the 7 Stage 1 phases (2,840 calls) in 30-minute chunks, downloading a checkpoint after each chunk, and stops
itself after 3 h 15 min or when Stage 1 is complete. If Colab disconnects, upload the newest checkpoint in cell 2 next
session and run this cell again — it continues where it stopped. Keep this tab open; allow multiple downloads.

In [20]:
# 7. Stage 1 run (Step B4)
import time, json
SESSION_BUDGET_MIN = 195
CHUNK_MIN = 30
start = time.time()
while True:
    left = SESSION_BUDGET_MIN - (time.time() - start) / 60
    if left < 3:
        print('Session budget used - keep the last checkpoint and continue next session.'); break
    get_ipython().system(f'python taskB/run_taskb.py --run --stage 1 --budget {int(min(CHUNK_MIN, left))}m')
    rc = get_ipython().user_ns.get('_exit_code', 0)
    save_checkpoint()
    if rc != 0:
        print('!! The runner stopped with an error (see above). Re-run cell 3, then this cell.'); break
    if json.load(open('results_taskB/progress_stage1.json'))['next_phase'] is None:
        print('STAGE 1 COMPLETE'); break

>> Stage 1: budget 30m00s, 1 worker(s)
== B1_baseline: already complete (632 calls)
== B1_S1_rubric: already complete (316 calls)
== B1_S2_fewshot: already complete (316 calls)
== B1_S3_steps: already complete (316 calls)
== B1_S4_accflu: already complete (316 calls)
== B1_pairwise: already complete (224 calls)
== B1_repeat_T07: 108/720 done, 612 to go
   [B1_repeat_T07] 25/612 calls  |  1.7 s/call  |  phase ETA 16m25s  |  session time left 29m17s
   [B1_repeat_T07] 50/612 calls  |  2.7 s/call  |  phase ETA 25m27s  |  session time left 27m44s
   [B1_repeat_T07] 75/612 calls  |  2.4 s/call  |  phase ETA 21m09s  |  session time left 27m02s
   [B1_repeat_T07] 100/612 calls  |  3.1 s/call  |  phase ETA 26m08s  |  session time left 24m53s
   [B1_repeat_T07] 125/612 calls  |  3.1 s/call  |  phase ETA 25m11s  |  session time left 23m32s
   [B1_repeat_T07] 150/612 calls  |  2.7 s/call  |  phase ETA 20m40s  |  session time left 23m17s
   [B1_repeat_T07] 175/612 calls  |  2.9 s/call  |  phase ET

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

STAGE 1 COMPLETE


In [21]:
# 8. Manual checkpoint (run before closing the tab)
save_checkpoint()

Checkpoint: /content/sprint53_taskB_run_20261003_0419.zip (2.2 MB) - downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>